Part 1 — the rate Source


In [0]:

###Q1. Create a streaming DataFrame using rate, 5 rows/sec.

df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()



In [0]:
### create volume
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.streaming_demo")


DataFrame[]

In [0]:
### create input folder

dbutils.fs.mkdirs("/Volumes/workspace/default/streaming_demo/input/")

True

In [0]:
### schema 
schema = "employee_id INT, name STRING, department STRING, salary DOUBLE"

In [0]:
##Q2. Start the query writing to an in-memory sink. What outputMode?
query = df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("rate_demo") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/rate_demo_checkpoint/") \
    .start()




In [0]:

%sql
---Query rate_demo twice
SELECT COUNT(*) FROM rate_demo;

COUNT(*)
2


In [0]:
##Streaming read of JSON files from the landing folder

input_path = "/Volumes/workspace/default/streaming_demo/input/"

df_stream = spark.readStream \
    .format("json") \
    .schema(schema) \
    .load(input_path)

df_stream.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)



In [0]:
query = df_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/checkpoint/") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("workspace.default.streaming_output")

In [0]:
%sql
SELECT COUNT(*) 
FROM workspace.default.streaming_output;

COUNT(*)
16


In [0]:
###display() on the streaming DataFrame directly. How does it differ?

display(
    df_stream,
    checkpointLocation="/Volumes/workspace/default/streaming_demo/display_checkpoint/"
)

Checkpointing to /Volumes/workspace/default/streaming_demo/display_checkpoint/
